# Entity graph: people, organizations, topics

Two channels of participation:
- **spoke** at a meeting (speakers table; solid edges, dot nodes)
- **wrote in** (emails bound into the minutes, parsed from attachment text; dashed edges; diamond nodes if email-only)

Names are **resolved**: honorifics stripped ("Mr. Steve Chesler" = "Steve Chesler")
and near-identical variants fuzzy-merged with a first-name guard (catches the
"Chesler"/"Chester" OCR split without merging the two different Thompsons).
The merge table prints below — review it; false merges can be vetoed in
`MANUAL_SPLIT`.

In [1]:
import json, re
from pathlib import Path
from collections import Counter

import duckdb
import pandas as pd
from pyvis.network import Network
from rapidfuzz import fuzz

con = duckdb.connect()
for t in ("meetings", "speakers"):
    con.execute(f"CREATE VIEW {t} AS SELECT * FROM '../data/db/{t}.parquet'")

sp = con.execute("""
    SELECT s.name, s.affiliation, s.topic, s.position, m.date::VARCHAR AS date
    FROM speakers s JOIN meetings m USING (meeting_id)
    WHERE s.name IS NOT NULL
""").df()
sp["channel"] = "spoke"
print(len(sp), "speaker records")

1360 speaker records


In [2]:
# --- email writers, parsed from the full text of every meeting ---
_meet = json.loads(Path("../data/meetings.json").read_text())["meetings"]
_sha2mid = {f["sha256"]: mid for mid, m in _meet.items() for f in m["files"]}
_texts = {}
for _tf in Path("../data/interim/text").glob("*.json"):
    _mid = _sha2mid.get(_tf.stem)
    if _mid:
        _texts[_mid] = _texts.get(_mid, "") + " " + " ".join(json.loads(_tf.read_text()))

EMAIL_RE = re.compile(
    r"From: ([A-Z][A-Za-z .\u2019'-]{3,40}?) Sent: .{0,120}?"
    r"Subject: (?:\[EXTERNAL\] )?(.{5,90}?)(?: CAUTION:| Date:| From:)"
)
rows = []
for mid, t in _texts.items():
    t = " ".join(t.split())  # regex "." must not be broken by page newlines
    for name, subject in EMAIL_RE.findall(t):
        rows.append({"name": name.strip(), "topic": subject.strip(),
                     "date": _meet[mid]["date"], "meeting_id": mid})
em = pd.DataFrame(rows).drop_duplicates(["name", "topic", "meeting_id"])
em["channel"] = "email"
em["affiliation"] = None
em["position"] = "unclear"  # emails don't carry an extracted position
print(f"{len(em)} distinct email submissions from {em.name.nunique()} names")

196 distinct email submissions from 181 names


In [3]:
# --- entity resolution across both channels ---
HONORIFICS = re.compile(
    r"^(mr|ms|mrs|dr|hon|rabbi|rev|councilmember|council member|assemblymember)\.? ",
    re.IGNORECASE)

def strip_name(n):
    n = " ".join(str(n).split())
    prev = None
    while prev != n:
        prev, n = n, HONORIFICS.sub("", n)
    return n.strip()

MANUAL_SPLIT = set()   # add ("name a", "name b") pairs here to veto a merge

all_names = pd.concat([sp.name, em.name]).map(strip_name)
counts = Counter(all_names)
names = sorted(counts, key=counts.get, reverse=True)

canon = {}
for n in names:
    low = n.lower()
    match = None
    for c0 in canon.values():
        cl = c0.lower()
        if (low, cl) in MANUAL_SPLIT or (cl, low) in MANUAL_SPLIT:
            continue
        f1, f2 = low.split()[0], cl.split()[0]
        first_ok = f1 == f2 or f1.startswith(f2) or f2.startswith(f1)
        if first_ok and fuzz.ratio(low, cl) >= 90:
            match = c0
            break
    canon[n] = match or n  # most frequent variant wins (names sorted by count)

merges = {k: v for k, v in canon.items() if k != v}
print(f"{len(merges)} variants merged:")
for k, v in sorted(merges.items()):
    print(f"  {k!r} -> {v!r}")

sp["canon"] = sp.name.map(lambda n: canon[strip_name(n)])
em["canon"] = em.name.map(lambda n: canon[strip_name(n)])

54 variants merged:
  'Aaron Kreiswirth' -> 'Aaron Kresiwirth'
  'Alex Rodriguez' -> 'Alexis Rodriguez'
  'Alvin Pena' -> 'Alvin Peña'
  'Barb Hartel' -> 'Barb Hertel'
  'Barbara Williams' -> 'Barbara J. Williams'
  'Barry Dross' -> 'Barry Druss'
  'Ben Solitaire' -> 'Ben Solataire'
  'Brent Bouenzi' -> 'Brent Bovenzi'
  'Brett Harman' -> 'Brett Herman'
  'Denny Tompkins' -> 'Denny Tomkins'
  'Elisha Fye' -> 'Elish Fye'
  'Eric Radesky' -> 'Eric Radeszky'
  'Eric Radezky' -> 'Eric Radeszky'
  'Francois Vaxelaire' -> 'Francoise Vaxelaire'
  'Francoise Oliva' -> 'Francoise Olivas'
  'Francoise Olivaz' -> 'Francoise Olivas'
  'Francoise Olivias' -> 'Francoise Olivas'
  'Gina Barros' -> 'Gina Barrows'
  'Glen Aaron' -> 'Glenn Aaron'
  'Heather Neufield' -> 'Heather Neufeld'
  'Ian Rasmussen' -> 'Ian Rasmussan'
  'Jane Clarke' -> 'Jane Clark'
  'Jeff Csicsec' -> 'Jeff Csicsek'
  'Jeff Csiczek' -> 'Jeff Csicsek'
  'Jeff Csisek' -> 'Jeff Csicsek'
  'Jessame Hannus' -> 'Jessame Mannus'
  'Kevi

In [4]:
# --- topic classification (same rules, both channels) ---
TOPIC_RULES = [
    ("Monitor Point", ["monitor point", "40 quay", "56 quay"]),
    ("Bushwick Inlet Park", ["bushwick inlet"]),
    ("River Ring", ["river ring"]),
    ("TAO / nightclubs", ["11-25 franklin", "tao group", "franklin bk",
                          "nightclub", "deuces"]),
    ("Nightlife / SLA", ["liquor", "sla ", "nightlife", "bar ", "applicant"]),
    ("National Grid / Newtown Creek", ["national grid", "newtown creek",
                                       "pipeline", "superfund"]),
    ("Cannabis", ["cannabis", "dispensary", "smoke shop"]),
    ("Parks (other)", ["park", "playground", "open space", "mcgolrick", "mcgorlick"]),
    ("Streets / transport", ["street", "bike", "traffic", "bus ", "g train",
                             "bqe", "mcguinness"]),
    ("Housing / land use", ["rezon", "housing", "development", "ulurp", "landmark"]),
]

def classify(topic):
    low = (topic or "").lower()
    for label, keys in TOPIC_RULES:
        if any(k in low for k in keys):
            return label
    return None

both = pd.concat([sp, em], ignore_index=True)
both["entity"] = both.topic.map(classify)
both["aff"] = (both.affiliation.str.strip().str.title()
               .replace({"Resident": None, "Residents": None}))
classified = both.dropna(subset=["entity"])
print(classified.groupby("channel").size())

channel
email    160
spoke    861
dtype: int64


In [5]:
MIN_PERSON = 2
MIN_ORG = 2
POS_COLOR = {"against": "#c0392b", "for": "#27ae60"}

def build_graph(df, height="750px"):
    net = Network(height=height, width="100%", notebook=True,
                  cdn_resources="in_line", bgcolor="#ffffff")
    people = df.groupby("canon").size()
    people = people[people >= MIN_PERSON].index

    for ent, n in df.entity.value_counts().items():
        net.add_node(f"T:{ent}", label=ent, color="#2d6a4f", shape="dot",
                     size=14 + min(n, 60) * 0.6, title=f"{n} records")

    pdf = df[df.canon.isin(people)]
    for name, grp in pdf.groupby("canon"):
        chans = set(grp.channel)
        shape = "dot" if "spoke" in chans else "diamond"
        span = f"{grp.date.min()[:4]}–{grp.date.max()[:4]}"
        net.add_node(f"P:{name}", label=name, color="#2b6cb0", shape=shape,
                     size=8 + len(grp),
                     title=f"{len(grp)} records ({', '.join(sorted(chans))}), {span}")
        for (ent, chan), egrp in grp.groupby(["entity", "channel"]):
            pos = Counter(egrp.position)
            top, _ = pos.most_common(1)[0]
            color = POS_COLOR.get(top, "#95a5a6")
            sample = "; ".join(egrp.topic.head(3).str[:60])
            net.add_edge(f"P:{name}", f"T:{ent}", value=len(egrp), color=color,
                         dashes=(chan == "email"),
                         title=f"[{chan}] {dict(pos)}\n{sample}")

    orgs = pdf.dropna(subset=["aff"]).groupby("aff").size()
    for org in orgs[orgs >= MIN_ORG].index:
        net.add_node(f"O:{org}", label=org, color="#e67e22", shape="box",
                     title="organization")
        for name in pdf[pdf.aff == org].canon.unique():
            if f"P:{name}" in net.get_nodes():
                net.add_edge(f"O:{org}", f"P:{name}", color="#f0c8a0", dashes=True)

    net.barnes_hut(gravity=-8000, central_gravity=0.4, spring_length=120)
    return net

net = build_graph(classified)
print(f"{len(net.get_nodes())} nodes, {len(net.get_edges())} edges")
net.show("entity_graph.html")

131 nodes, 199 edges
entity_graph.html


Cross-channel people (spoke AND wrote in) — often the most engaged:

In [6]:
chan_by_person = classified.groupby("canon").channel.agg(set)
cross = chan_by_person[chan_by_person.map(len) > 1].index.tolist()
classified[classified.canon.isin(cross)].groupby("canon").agg(
    records=("canon", "size"),
    topics=("entity", lambda s: ", ".join(sorted(set(s)))),
).sort_values("records", ascending=False)

,records,topics
canon,,
Libby Brennan,2,"Bushwick Inlet Park, TAO / nightclubs"
Michael Hofmann,2,"Bushwick Inlet Park, Housing / land use"
Sarah Magid,2,"Nightlife / SLA, TAO / nightclubs"


Focused view — waterfront saga only:

In [7]:
WATERFRONT = ["Monitor Point", "Bushwick Inlet Park", "River Ring",
              "TAO / nightclubs", "National Grid / Newtown Creek"]
net2 = build_graph(classified[classified.entity.isin(WATERFRONT)], height="650px")
net2.show("waterfront_graph.html")

waterfront_graph.html
